In [ ]:
%pip install tensorflow split-folders tensorflowjs matplotlib pillow

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import splitfolders
import matplotlib.pyplot as plt
import os
import shutil

dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)

base_dir = 'flower_dataset'
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)

splitfolders.ratio(data_dir, output=base_dir, seed=42, ratio=(.8, .1, .1))

train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

print("Data berhasil diunduh dan dibagi!")

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    shear_range=0.2,
    fill_mode='nearest'
)

test_val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=(150, 150), batch_size=32, class_mode='categorical'
)
val_gen = test_val_datagen.flow_from_directory(
    val_dir, target_size=(150, 150), batch_size=32, class_mode='categorical'
)
test_gen = test_val_datagen.flow_from_directory(
    test_dir, target_size=(150, 150), batch_size=32, class_mode='categorical', shuffle=False
)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(150, 150, 3), include_top=False, weights='imagenet'
)
base_model.trainable = False 

model = Sequential([
    base_model,
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),                                    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5), 
    Dense(5, activation='softmax') 
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("Arsitektur Model Siap!")

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
]

history = model.fit(train_gen, epochs=10, validation_data=val_gen, callbacks=callbacks)

test_loss, test_acc = model.evaluate(test_gen)
print(f"\n---> Akurasi Testing: {test_acc * 100:.2f}% <---")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Grafik Akurasi')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Grafik Loss')
plt.legend()
plt.show()

In [ ]:
import os

os.makedirs('submission/saved_model', exist_ok=True)
os.makedirs('submission/tflite', exist_ok=True)
os.makedirs('submission/tfjs_model', exist_ok=True)

model.export('submission/saved_model')
print("Model berhasil disimpan ke format SavedModel!")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('submission/tflite/model.tflite', 'wb') as f:
    f.write(tflite_model)
    
labels = '\n'.join(sorted(train_gen.class_indices.keys()))
with open('submission/tflite/label.txt', 'w') as f:
    f.write(labels)
print("Model berhasil disimpan ke format TF-Lite!")

In [ ]:
print("\n=== Evaluasi pada Testing Set ===")
test_loss, test_acc = model.evaluate(test_gen)
print(f"Akurasi Testing: {test_acc * 100:.2f}%")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Grafik Akurasi')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Grafik Loss')
plt.legend()
plt.show()

In [ ]:
import pathlib

os.makedirs('submission/tfjs_model', exist_ok=True)
os.makedirs('submission/tflite', exist_ok=True)
os.makedirs('submission/saved_model', exist_ok=True)

model.export('submission/saved_model')
print("Model berhasil disimpan ke format SavedModel!")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('submission/tflite/model.tflite', 'wb') as f:
    f.write(tflite_model)
    
labels = '\n'.join(sorted(train_gen.class_indices.keys()))
with open('submission/tflite/label.txt', 'w') as f:
    f.write(labels)
print("Model berhasil disimpan ke format TF-Lite!")